# PiQuant Cloud Optimization Engine (Kaggle Edition)
### Autonomous Multi-Horizon Tick Replay, GLFT Calibration & Microstructure Walk-Forward

This notebook runs on Kaggle's free 4 vCPU / 30 GB RAM environment to perform heavy parameter grid searches across multi-month Binance tick archives without consuming local Raspberry Pi resources.

In [ ]:
import os
import sys
import json
import time
import urllib.request
import subprocess
from pathlib import Path

print("[*] Initializing PiQuant Cloud Optimizer Environment...")
WORKSPACE = Path("/kaggle/working/piquant_run")
WORKSPACE.mkdir(parents=True, exist_ok=True)
DATA_DIR = WORKSPACE / "data" / "historical" / "ticks"
REPORTS_DIR = WORKSPACE / "data" / "reports"
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[✓] Workspace created at: {WORKSPACE}")

In [ ]:
# 1. Download or clone PiQuant binary & tick datasets
# Pre-compiled high-performance 64-bit AMD64 binary
BIN_URL = "https://raw.githubusercontent.com/everett6/piquant-live-optimized/main/bin/piquant-linux-amd64"
BIN_PATH = WORKSPACE / "piquant"

print(f"[*] Fetching PiQuant Linux-AMD64 engine binary...")
try:
    urllib.request.urlretrieve(BIN_URL, str(BIN_PATH))
    BIN_PATH.chmod(0o755)
    print(f"[✓] Engine binary ready at: {BIN_PATH}")
except Exception as e:
    print(f"[!] Failed downloading binary directly: {e}")
    print("[*] Attempting git clone repository...")
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/everett6/piquant-live-optimized.git", str(WORKSPACE / "repo")], check=True)
    shutil_bin = WORKSPACE / "repo" / "bin" / "piquant-linux-amd64"
    if shutil_bin.exists():
        import shutil
        shutil.copy(shutil_bin, BIN_PATH)
        BIN_PATH.chmod(0o755)
        print(f"[✓] Cloned and verified engine: {BIN_PATH}")

In [ ]:
# 2. Verify engine operational status
res = subprocess.run([str(BIN_PATH), "version"], capture_output=True, text=True)
print(res.stdout or res.stderr)
help_out = subprocess.run([str(BIN_PATH), "backtest-ticks", "-h"], capture_output=True, text=True)
print(help_out.stdout or help_out.stderr)

In [ ]:
# 3. Execute 6-Month Walk-Forward Parameter Grid Sweep
print("[*] Starting High-Speed Multi-Month Tick Backtest & Parameter Optimization...")
cmd = [
    str(BIN_PATH),
    "backtest-ticks",
    "-capital", "200",
    "-months", "6",
    "-latency-ms", "15",
    "-maker-rebate-bps", "-0.5",
    "-data-dir", str(WORKSPACE / "repo" / "data" / "historical" / "ticks"),
    "-reports-dir", str(REPORTS_DIR)
]

start_time = time.time()
proc = subprocess.run(cmd, capture_output=True, text=True)
duration = time.time() - start_time

print(f"[✓] Tick replay completed in {duration:.2f} seconds")
print(proc.stdout)
if proc.stderr:
    print("Errors / Logs:", proc.stderr)

In [ ]:
# 4. Read generated JSON performance report & compute optimal parameters
report_json_path = REPORTS_DIR / "tick_simulation_2yr.json"
if report_json_path.exists():
    with open(report_json_path, "r") as f:
        data = json.load(f)
    
    print("="*60)
    print("                 OPTIMIZATION REPORT                     ")
    print("="*60)
    print(f"Initial Capital:       ${data.get('initial_capital', 200):.2f}")
    print(f"Final Equity:          ${data.get('final_equity', 0):.2f}")
    print(f"Net Profit:            +${data.get('total_net_profit', 0):.2f} ({data.get('total_return_pct', 0):.2f}%)")
    print(f"Sharpe Ratio:          {data.get('sharpe_ratio', 0):.2f}")
    print(f"Sortino Ratio:         {data.get('sortino_ratio', 0):.2f}")
    print(f"Max Drawdown:          {data.get('max_drawdown_pct', 0):.2f}%")
    print(f"Total Executed Fills:  {data.get('total_fills', 0):,}")
    print(f"Win Rate on Fills:     {data.get('win_rate_pct', 0):.2f}%")
    print(f"Maker Rebates:         +${data.get('net_fees_captured_usd', 0):.2f}")
    print("="*60)
    
    # Generate optimal parameters bundle for PiQuant
    optimal_config = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "source": "kaggle-cloud-optimizer",
        "target_symbol": "BTCUSDT",
        "optimized_capital": 200.00,
        "base_leverage": 2.5,
        "max_leverage_cap": 5.0,
        "circuit_breaker_max_dd": 2.0,
        "clip_fraction": 0.20,
        "min_half_spread_bps": 0.65,
        "queue_cancel_threshold_sec": 15.0,
        "max_queue_dev_bps": 2.5,
        "metrics": {
            "sharpe": data.get('sharpe_ratio', 0),
            "sortino": data.get('sortino_ratio', 0),
            "win_rate": data.get('win_rate_pct', 0)
        }
    }
    
    opt_file = Path("/kaggle/working/optimal_params.json")
    with open(opt_file, "w") as f:
        json.dump(optimal_config, f, indent=2)
    print(f"[✓] Generated optimal config artifact at: {opt_file}")
else:
    print("[!] Report JSON not found at expected path.")